# 第12課 – 使用代理草稿本減少聊天歷史

本筆記本展示如何使用 Microsoft Agent 框架來管理長對話中的上下文。隨著對話變長，令牌數量會增加——最終超出模型的上下文窗口。我們通過<strong>上下文總結模式</strong>以及用於持久記憶的<strong>代理草稿本</strong>來解決這個問題。

## 你將學習到的內容：
1. <strong>為何上下文管理重要</strong>：了解令牌限制與上下文窗口
2. <strong>上下文感知代理</strong>：構建能管理自己對話上下文的代理
3. <strong>上下文總結模式</strong>：使用工具濃縮對話歷史
4. <strong>代理草稿本</strong>：在上下文減少後仍保持的持久記憶

## 先決條件：
- 設定有環境變量配置的 Azure OpenAI
- 理解先前課程中的基本代理概念


## 設定


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## 為何上下文管理很重要

每個大型語言模型（LLM）都有一個有限的<strong>上下文視窗</strong> —— 它在單次請求中可處理的最大標記數量。隨著多輪對話的進展：

- <strong>標記數量會隨著每條用戶訊息和助理回覆線性增加</strong>。
- <strong>提示標記佔成本主導</strong>，因為整個歷史記錄每輪都會被重新傳送。
- 最終對話會<strong>超出上下文視窗</strong>，模型會截斷或出錯。

### 上下文管理策略

| 策略 | 運作方式 | 折衷 |
|---|---|---|
| <strong>截斷</strong> | 丟棄最舊訊息 | 失去早期上下文 |
| <strong>摘要</strong> | 將較舊訊息濃縮成摘要 | 損失部分細節，但保留重點 |
| **草稿本 / 外部記憶** | 將關鍵事實存放於對話外 | 需調用工具，但即使減少也可保存 |

本筆記本結合了<strong>摘要</strong>與<strong>草稿工具</strong>，讓代理在對話歷史被濃縮時仍能維持連貫性。


## 建立一個具備情境感知的代理程式


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## 模擬一段長時間對話

讓我們通過一段多回合對話來看看上下文如何累積。代理人應該在多輪對話中保留關鍵細節（偏好、預算、旅行日期），並展現連貫性。


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

注意代理如何保留早期對話的上下文 — 它知道日本、壽司、寺廟、攝影、3000美元預算、單人旅行，以及四月的時間範圍。在短對話中這很有效，但隨著對話增加，重新傳送完整歷史會變得昂貴。

讓我們繼續更多輪對話，看看上下文的累積效果：


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## 上下文總結模式

隨著對話的增長，我們可以使用 <strong>總結工具</strong> 將累積的上下文濃縮成緊湊的格式。代理會調用此工具來記錄關鍵偏好，這樣即使較舊的訊息被丟棄，重要資訊仍能被保留。

此模式是更複雜歷史縮減的基礎構建塊：
1. 代理識別對話中的關鍵事實
2. 調用總結工具將其持久化
3. 可以安全移除較舊的訊息，因為總結捕捉了重要內容

下方定義了一個 `summarize_preferences` 工具，代理可以調用它來記錄已學習內容的精簡摘要。


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## 摘要

在本課程中，你學習了如何使用 Microsoft Agent Framework 來管理長時間運行的代理對話中的上下文：

### 主要概念
- <strong>上下文視窗是有限的</strong> — 對話記錄中的每個標記都會產生成本並計入限制。
- <strong>摘要工具</strong> 讓代理能將累積的上下文濃縮成緊湊的摘要，減少標記用量同時保留重要資訊。
- <strong>代理便箋</strong> 提供持久的外部記憶，可保存任何對話縮減後的內容。

### 你所建立的內容
- 一個 <strong>上下文感知代理</strong>，能維持多輪對話的連續性
- 一個 <strong>摘要工具</strong>（`summarize_preferences`），能以緊湊格式記錄關鍵的使用者細節
- 一個展示上下文保留與變更處理的 <strong>多輪對話</strong>

### 實際應用
- <strong>客戶服務機器人</strong>：記住長時間支援對話中的偏好
- <strong>個人助理</strong>：追蹤進行中的專案，無需重複解釋上下文
- <strong>教育輔導員</strong>：維持學生多次互動中的學習進度

### 後續步驟
- 實作擁有文件持久化功能的完整便箋工具
- 在摘要後加入自動歷史截斷
- 結合向量資料庫以實現語意記憶檢索
- 建立能在數日後以完整上下文恢復對話的代理


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
